# 03 · Deep-learning baselines (Milestone 2)

**Pilot region:** Gulf of Mannar – Palk Bay (`IN-GOM`). **Data:** 60-day synthetic simulation (Mode A).

What this notebook does:
1. builds (or loads cached) a 60-day synthetic dataset — 62 vessels, four suspicious carrier networks;
2. turns tracks into **6-hour vessel windows** (36 × 10-min steps with an explicit *observed* mask; silent steps are never interpolated);
3. makes **vessel-disjoint** and **time-disjoint** splits (no vessel / voyage leakage);
4. evaluates a transparent **rule-based reference** and trains three **neural baselines** — MLP (window aggregates), bidirectional GRU and dilated TCN (step sequences) — with 3 seeds each;
5. reports PR-AUC, precision/recall/F1 at a validation-calibrated threshold, **false alerts per 1,000 vessel-hours**, **event-level recall**, confusion matrices, learning curves, and a **false-alert gallery** (anchorage, MPA transit, receiver outage, loitering carriers, …).

**Runtime:** CPU only is fine — about 3 min for data + about 8 min for training (3 seeds × 3 models × 2 splits). A GPU (Runtime → Change runtime type → T4) speeds up the GRU/TCN cells; nothing requires it. Set `FAST_MODE = True` for a 1-seed run (~3 min).

> ⚠️ **Synthetic demo data.** Labels are planted simulation events (`synthetic_demo_suspicious`). Scores mean *investigation priority*, never illegality. The rule reference is **not** the contribution; the neural baselines exist to be beaten by the temporal graph model in Milestone 3.

In [ ]:
# ---- Setup: works in Google Colab (CPU is enough) and locally --------------------
import os, sys, subprocess, pathlib, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Dan-droid-A-113/IUU-Fishing-GNN-India"   # clones the code; set to "" to upload the zip instead
USE_DRIVE = False      # True = keep project + outputs on Google Drive (restart-safe)

if IN_COLAB:
    PROJECT_DIR = pathlib.Path("/content/india-fishing-gnn")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn")
    if not (PROJECT_DIR / "src").exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
        else:
            # accepts renamed downloads such as "india-fishing-gnn (1).zip"
            zips = sorted(glob.glob("/content/india-fishing-gnn*.zip"), key=os.path.getmtime)
            if not zips:
                from google.colab import files
                print("Project zip not found - choose india-fishing-gnn.zip from your computer:")
                uploaded = files.upload()   # file picker appears below this cell
                zips = [os.path.abspath(n) for n in uploaded if n.endswith(".zip")]
            if not zips:
                raise FileNotFoundError("No project zip uploaded.")
            print("Extracting", zips[-1])
            zipfile.ZipFile(zips[-1]).extractall(PROJECT_DIR.parent)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-colab.txt")], check=True)
else:
    PROJECT_DIR = pathlib.Path.cwd()
    while not (PROJECT_DIR / "src").exists() and PROJECT_DIR != PROJECT_DIR.parent:
        PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR, "| Colab:", IN_COLAB)

## 0 · Configuration

In [ ]:
import json, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from src.config import TrainConfig, WindowConfig, FIGURE_DIR, METRIC_DIR, SYNTHETIC_CAVEAT
from src.experiments import prepare_m2_data, make_splits, run_baselines, comparison_table, best_run, M2_PROC_DIR
from src.baselines import get_device, RULES
from src.windows import STEP_FEATURES, STATIC_FEATURES
from src import visualization as viz

FAST_MODE = False                  # True = 1 seed per model (quick look)
FORCE_REBUILD = False              # True = regenerate synthetic data and windows
CFG = TrainConfig(seeds=(0,) if FAST_MODE else (0, 1, 2))
DEVICE = get_device()              # GPU if available, otherwise CPU
torch.set_num_threads(max(1, torch.get_num_threads()))
for p in (FIGURE_DIR, METRIC_DIR):
    p.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| seeds:", CFG.seeds)
print(SYNTHETIC_CAVEAT)

## 1 · Data: 60-day simulation → preprocessing → 6-h windows (cached)

Each stage is cached under `data/synthetic_m2/` and `data/processed_m2/`, so a restarted runtime resumes quickly.

In [ ]:
d = prepare_m2_data(force=FORCE_REBUILD)
lab = d["labels"]
display(pd.Series(json.loads((M2_PROC_DIR / "preprocess_report.json").read_text()), name="value").to_frame().T)
ev = d["events"]
display(ev.groupby(["event_type", "synthetic_demo_label"]).size().rename("planted events").to_frame())
display(lab.context.value_counts().rename("windows").to_frame())

In [ ]:
print(f"Sequence tensor: {d['X'].shape}  (windows x 10-min steps x features)")
print("Step features :", ", ".join(STEP_FEATURES))
print("Static        :", ", ".join(STATIC_FEATURES))
print(f"MLP aggregates: {d['agg'].shape[1]} columns ->", ", ".join(d["agg"].columns[:12]), "...")

## 2 · Leakage-safe splits

* **Vessel-disjoint** — whole vessels per split. Vessels in a *labelled* interaction (planted rendezvous, fleet pairs, tug assists — tugs excluded because they meet almost every ship) are grouped, so each suspicious carrier network sits entirely in one split (2 train / 1 validation / 1 test). Over 60 days, *unlabelled* proximity links chain almost all vessels together, so they cannot be used for grouping; the number of such cross-split contacts is reported instead.
* **Time-disjoint** — days 1–40 train, 41–50 validation, 51–60 test, 6-h embargo; windows touching a voyage that straddles a boundary are dropped.
* **Region-disjoint** — not possible yet (one pilot region).

In [ ]:
from src.splits import cross_split_contacts
splits = make_splits(d)
for name, sp in splits.items():
    display(pd.crosstab(pd.Series(sp, name=name), lab.y.rename("synthetic_demo_suspicious")))
vsplit = splits["vessel_disjoint"]
print("cross-split sustained-proximity pairs touching the test vessels:",
      cross_split_contacts(lab, vsplit, d["pairs"]))

## 3 · Rule-based reference (transparent, not learned)

In [ ]:
display(pd.Series(RULES, name="rule").to_frame())

## 4 · Train and evaluate (rule + MLP + GRU + TCN, all seeds, both splits)

Loss: class-weighted binary cross-entropy. Optimiser: AdamW + ReduceLROnPlateau. Early stopping on **validation PR-AUC**; the alert threshold is also chosen on validation (max F1) and then applied unchanged to the test set. Checkpoints go to `outputs/checkpoints/`. *GPU optional — CPU ~8 min.*

In [ ]:
res = run_baselines(d, splits, CFG, device=DEVICE)
res["runs"].to_csv(METRIC_DIR / "m2_runs_all_seeds.csv", index=False)
table = comparison_table(res["runs"])
table.to_csv(METRIC_DIR / "m2_model_comparison.csv")
display(table)

## 5 · Curves and confusion matrices

In [ ]:
fig = viz.plot_pr_curves(res, lab, FIGURE_DIR / "m2_pr_curves.png"); plt.show()
for sp in splits:
    fig = viz.plot_confusion_matrices(res, lab, sp, FIGURE_DIR / f"m2_confusion_{sp}.png"); plt.show()

In [ ]:
for sp in splits:
    fig = viz.plot_learning_curves(res, sp, FIGURE_DIR / f"m2_learning_curves_{sp}.png"); plt.show()

## 6 · Event-level detection and performance by vessel type

In [ ]:
rows = []
for (sp, m, seed), p in res["preds"].items():
    e = p["events"]
    if len(e):
        for t, g in e.groupby("event_type"):
            rows.append({"split": sp, "model": m, "seed": seed, "event_type": t, "events": len(g),
                         "recall": g.detected.mean()})
ev_tab = (pd.DataFrame(rows).groupby(["split", "event_type", "model"]).recall.mean()
          .unstack("model").round(2))
ev_tab.to_csv(METRIC_DIR / "m2_event_recall_by_type.csv")
display(ev_tab)

from src.evaluation import binary_metrics
rows = []
for sp in splits:
    for m in res["runs"][res["runs"].split == sp].model.unique():
        p = best_run(res, sp, m)
        L = lab.iloc[p["test_idx"]]
        for vt, idx in L.groupby("vessel_type").indices.items():
            mm = binary_metrics(L.y.values[idx], p["test"][idx], p["thr"], L.active_hours.values[idx])
            rows.append({"split": sp, "model": m, "vessel_type": vt, "windows": len(idx), "positives": mm["positives"],
                         "recall": mm["recall"], "false_alerts_per_1000_vessel_h": mm["false_alerts_per_1000_vessel_h"]})
vt_tab = pd.DataFrame(rows)
vt_tab.to_csv(METRIC_DIR / "m2_by_vessel_type.csv", index=False)
display(vt_tab[vt_tab.split == "vessel_disjoint"].pivot(index="vessel_type", columns="model",
        values="false_alerts_per_1000_vessel_h").round(1))

## 7 · Where do false alerts come from? (hard negatives)

In [ ]:
for sp in splits:
    t = viz.fp_by_context(res, lab, sp)
    t.to_csv(METRIC_DIR / f"m2_false_alerts_by_context_{sp}.csv", index=False)
    fig = viz.plot_fp_by_context(t, sp, FIGURE_DIR / f"m2_false_alerts_by_context_{sp}.png"); plt.show()

In [ ]:
r = res["runs"]
best_nn = (r[(r.split == "vessel_disjoint") & (r.model != "Rule (reference)")]
           .groupby("model").pr_auc.mean().idxmax())
for m in dict.fromkeys([best_nn, "TCN"]):
    fig = viz.plot_fp_gallery(d, best_run(res, "vessel_disjoint", m), "vessel_disjoint", m,
                              FIGURE_DIR / f"m2_false_alert_gallery_{m}.png")
    if fig is not None:
        plt.show()

## 8 · Save summary

In [ ]:
summary = {
    "milestone": 2, "data_mode": "A_synthetic (60 days)", "caveat": SYNTHETIC_CAVEAT,
    "n_windows": int(len(lab)), "positives": int(lab.y.sum()), "device": str(DEVICE),
    "train_config": vars(CFG), "window_config": vars(WindowConfig()),
    "comparison_mean": res["runs"].groupby(["split", "model"])[["pr_auc", "f1", "precision", "recall",
        "false_alerts_per_1000_vessel_h", "event_recall"]].mean().round(4).reset_index().to_dict("records"),
}
(METRIC_DIR / "m2_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("metrics:", *sorted(p.name for p in METRIC_DIR.glob("m2_*")), sep="\n  ")
print("figures:", *sorted(p.name for p in FIGURE_DIR.glob("m2_*")), sep="\n  ")

## What Milestone 2 can and cannot conclude

**Can conclude (synthetic pilot)**
* The full learning pipeline works end to end — windowing, leakage-safe splits, class-weighted training with early stopping, validation-calibrated thresholds, window- and event-level metrics — on CPU in Colab.
* Planted suspicious episodes are recoverable from AIS kinematics, explicit gap events, boundary/MPA context and neighbour summaries. Compare models in the table above; the MLP on hand-built window aggregates (including gap-event features) is the strongest baseline, and the TCN beats the GRU among sequence models.
* **Hard negatives dominate false alerts:** benign AIS gaps (transmitter faults, coverage holes, receiver outages) and loitering carriers are the main confusions, which is exactly the context a graph model with neighbour messages and reception evidence should help with.

**Cannot conclude**
* Anything about real vessels or real Indian waters — labels are planted by the simulator, and the synthetic suspicious behaviours are partly "rule-shaped", which flatters the rule reference.
* How the models would rank on real AIS, with real class imbalance and label noise.
* Region transfer (only one pilot region exists).

**Next — Milestone 3:** a temporal graph-attention network (vessels = nodes, encounters and possible dark encounters = edges) with multi-task heads and a neural gap-trajectory decoder, compared against these baselines on the same splits.